# INITIAL IMPORT

In [80]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import os
from src.config import Configuration


CONFIG = Configuration(

)

# VIEW

In [8]:
import torch
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider

def plot_patient(tensor_path):
    # Load stacked tensor (4, X, Y, Z)
    volume_tensor = torch.load(tensor_path, weights_only=True)
    volume_data = volume_tensor.numpy()
    
    modalities = ['T1', 'T1-Gd', 'T2', 'FLAIR']

    def _update_plot(slice_idx):
        # Create a 1x4 grid
        fig = make_subplots(
            rows=1, cols=4, 
            subplot_titles=modalities,
            horizontal_spacing=0.02
        )

        for i, mod_name in enumerate(modalities):
            img_slice = volume_data[i, :, :, slice_idx]
            
            fig.add_trace(
                go.Heatmap(
                    z=img_slice,
                    colorscale='Gray',
                    showscale=False,
                    transpose=True
                ),
                row=1, col=i+1
            )

        fig.update_layout(
            title_text=f"Patient: {os.path.basename(tensor_path)} | Axial Slice: {slice_idx}",
            width=1200, # Wide enough for 4 images
            height=400,
            margin=dict(l=10, r=10, t=50, b=10)
        )
        
        # Ensure consistent aspect ratios across all subplots
        fig.update_xaxes(visible=False)
        fig.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)
        
        fig.show()

    # Slider for depth (Z-axis)
    interact(
        _update_plot, 
        slice_idx=IntSlider(
            min=0, 
            max=volume_data.shape[3]-1, 
            step=1, 
            value=volume_data.shape[3]//2,
            description='Slice Z:'
        )
    )

# Usage
# plot_all_modalities_plotly("data/tensors/UPENN-GBM-00001.pt")

In [ ]:
file_path = os.path.join(CONFIG.mr_nf_tensors, "UPENN-GBM-00465.pt")
plot_patient(file_path)

interactive(children=(IntSlider(value=77, description='Slice Z:', max=154), Output()), _dom_classes=('widget-i…

In [ ]:
file_path = os.path.join(CONFIG.mr_nf_tensors, "UPENN-GBM-00001.pt")
plot_patient(file_path)

interactive(children=(IntSlider(value=77, description='Slice Z:', max=154), Output()), _dom_classes=('widget-i…

# Data loader

In [113]:
import pandas as pd

df = pd.read_csv(CONFIG.mr_data)
# df = df[df['ID'].str.endswith('_11')].copy()

len(df)

671

In [114]:
# 1. Create a temporary column with the Patient Root ID (e.g., UPENN-GBM-00612)
df['PatientRoot'] = df['ID'].str.split('_').str[0]
# 2. Sort by ID so that _11 comes before _21 for the same patient
df = df.sort_values(by='ID')
# 3. Drop duplicates based on the Root ID, keeping the first one found
# This keeps _11 if it exists; otherwise, it keeps _21.
df = df.drop_duplicates(subset='PatientRoot', keep='first')
# 4. Clean up the temporary column
df = df.drop(columns=['PatientRoot'])

len(df)

630

In [115]:
df[df["Survival_from_surgery_days_UPDATED"] == 'Not Available']


,ID,Gender,Age_at_scan_years,Survival_from_surgery_days_UPDATED,Survival_Status,Survival_Censor,IDH1,MGMT,KPS,GTR_over90percent,Time_since_baseline_preop,PsP_TP_score
22,UPENN-GBM-00023_11,M,55.57,Not Available,Lost to Follow-up,1341,Wildtype,Not Available,Not Available,N,0,NaN
76,UPENN-GBM-00071_11,F,37.13,Not Available,Alive,6129,Mutated,Not Available,Not Available,Y,0,NaN
91,UPENN-GBM-00085_11,M,48.18,Not Available,Alive,6135,Wildtype,Not Available,Not Available,Y,0,NaN
133,UPENN-GBM-00123_11,F,24.61,Not Available,Lost to Follow-up,488,Mutated,Not Available,Not Available,Not Available,0,NaN
159,UPENN-GBM-00142_11,M,67.68,Not Available,Deceased - uncertain date of death,142,Wildtype,Not Available,Not Available,N,0,NaN
210,UPENN-GBM-00188_11,M,44.46,Not Available,Lost to Follow-up,4049,NOS/NEC,Not Available,Not Available,Y,0,NaN
272,UPENN-GBM-00248_11,F,45.71,Not Available,Alive,6160,Wildtype,Not Available,Not Available,Y,0,NaN
282,UPENN-GBM-00258_11,M,50.71,Not Available,Alive,5977,NOS/NEC,Not Available,Not Available,Y,0,NaN
294,UPENN-GBM-00269_11,M,56.21,Not Available,Alive,3743,Wildtype,Methylated,Not Available,Y,0,NaN
301,UPENN-GBM-00276_11,M,18.65,Not Available,Lost to Follow-up,4491,NOS/NEC,Not Available,Not Available,Y,0,NaN


In [116]:
# 1. Coerce invalid strings to NaN
df['Survival_from_surgery_days_UPDATED'] = pd.to_numeric(df['Survival_from_surgery_days_UPDATED'], errors='coerce')

# 2. Drop rows where survival is NaN
df = df.dropna(subset=['Survival_from_surgery_days_UPDATED'])

# 3. Now safe to convert to int
df['Survival_from_surgery_days_UPDATED'] = df['Survival_from_surgery_days_UPDATED'].astype(int)

df["Survival_from_surgery_days_UPDATED"].describe()

count     603.000000
mean      510.112769
std       518.107626
min         3.000000
25%       170.500000
50%       384.000000
75%       630.500000
max      6109.000000
Name: Survival_from_surgery_days_UPDATED, dtype: float64

In [117]:
df['Gender'] = df['Gender'].map({'F': 0, 'M': 1})

df['Gender'].value_counts()

Gender
1    360
0    243
Name: count, dtype: int64

In [118]:
df['Age_at_scan_years'].describe()


count    603.000000
mean      63.176418
std       11.985623
min       20.740000
25%       56.125000
50%       63.580000
75%       71.640000
max       88.500000
Name: Age_at_scan_years, dtype: float64

In [119]:
df['IDH1'].value_counts()

IDH1
Wildtype    496
NOS/NEC      94
Mutated      13
Name: count, dtype: int64

In [120]:
df['MGMT'].value_counts()

MGMT
Not Available    313
Unmethylated     159
Methylated       106
Indeterminate     25
Name: count, dtype: int64

In [121]:
df['GTR_over90percent'].value_counts()

GTR_over90percent
Y                 347
N                 204
Not Available      34
Not Applicable     18
Name: count, dtype: int64

In [122]:
df['KPS'].value_counts()


KPS
Not Available    529
90                37
80                18
60                 6
70                 5
100                5
40                 2
30                 1
Name: count, dtype: int64

In [123]:
df['KPS'] = pd.to_numeric(df['KPS'], errors='coerce')
# Define the bins: VERY LITTLE DATA, so keep it in just 3 beans
# Groups: 0-70 (Impaired), 80-100 (Functional), NaN (Unknown)
def bin_kps(score):
    if pd.isna(score):
        return 'Unknown'
    elif score <= 80:
        return 'High_KPS'
    else:
        return 'Low_KPS'
df['KPS'] = df['KPS'].apply(bin_kps)
        

df['IDH1'] = df['IDH1'].replace(['NOS/NEC'], 'Unknown').fillna('Unknown')

# 2. Standardize MGMT (Map Indeterminate and Not Available to 'Unknown')
df['MGMT'] = df['MGMT'].replace(['Not Available', 'Indeterminate'], 'Unknown').fillna('Unknown')

# 3. Standardize GTR (Map Not Available/Applicable to 'Unknown')
df['GTR_over90percent'] = df['GTR_over90percent'].replace(['Not Available', 'Not Applicable'], 'Unknown').fillna('Unknown')

# 4. Create Dummies for these columns
# This will create columns like: MGMT_Methylated, MGMT_Unmethylated, MGMT_Unknown
clinical_cols = ['KPS', 'IDH1', 'MGMT', 'GTR_over90percent']
df = pd.get_dummies(df, columns=clinical_cols, prefix=clinical_cols)

# 5. Convert any remaining boolean dummies to int (0/1)
# This ensures your neural network receives a numeric matrix
for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)
        

In [124]:
df.head()


,ID,Gender,Age_at_scan_years,Survival_from_surgery_days_UPDATED,Survival_Status,Survival_Censor,Time_since_baseline_preop,PsP_TP_score,KPS_High_KPS,KPS_Low_KPS,KPS_Unknown,IDH1_Mutated,IDH1_Unknown,IDH1_Wildtype,MGMT_Methylated,MGMT_Unknown,MGMT_Unmethylated,GTR_over90percent_N,GTR_over90percent_Unknown,GTR_over90percent_Y
0,UPENN-GBM-00001_11,0,52.16,960,Deceased,Not Available,0,NaN,0,0,1,0,0,1,0,1,0,0,0,1
1,UPENN-GBM-00002_11,0,61.30,291,Deceased,Not Available,0,NaN,0,0,1,0,0,1,0,1,0,0,0,1
2,UPENN-GBM-00003_11,1,42.82,2838,Deceased,Not Available,0,NaN,0,0,1,0,0,1,0,1,0,0,0,1
3,UPENN-GBM-00004_11,1,33.43,623,Deceased,Not Available,0,NaN,0,0,1,0,1,0,0,1,0,0,0,1
4,UPENN-GBM-00005_11,1,53.33,1143,Deceased,Not Available,0,NaN,0,0,1,0,0,1,0,1,0,0,0,1


In [132]:
from src.data import UPennGBMDataset

data = UPennGBMDataset(CONFIG)

In [134]:
df['PatientRoot'] = df['ID'].str.split('_').str[0]


In [135]:
df['PatientRoot']

0      UPENN-GBM-00001
1      UPENN-GBM-00002
2      UPENN-GBM-00003
3      UPENN-GBM-00004
4      UPENN-GBM-00005
            ...       
666    UPENN-GBM-00626
667    UPENN-GBM-00627
668    UPENN-GBM-00628
669    UPENN-GBM-00629
670    UPENN-GBM-00630
Name: PatientRoot, Length: 603, dtype: object